# 02 数据质量审计与清洗

这个文件解决实际风控分析前的数据质量问题。

原则：不要一看到异常值就删除。先判断它是否可能代表风险，再决定保留、标记或置为空。


## 1. 本阶段产出

本阶段产出两个文件：

1. `credit_risk_cleaned.csv`：后续风险因子分析使用的主数据；
2. `cleaning_audit_summary.csv`：清洗依据，用于解释为什么这样处理。


## 2. 导入工具、设置路径、读取数据


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

# 项目根目录：如果你的项目文件夹换位置，只需要改这一行。
PROJECT_ROOT = Path(r"E:\DataAnalysis\DataAnalysisRoad\Projects\01_Credit_Risk_Analysis")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cs-training.csv"

RAW_DATA_PATH


In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)

# 原始字段名太长，先改成工作中更好读的字段名。
rename_cols = {
    "id": "customer_id",
    "SeriousDlqin2yrs": "is_bad_customer",
    "RevolvingUtilizationOfUnsecuredLines": "credit_utilization",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "past_due_30_59_count",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "open_credit_count",
    "NumberOfTimes90DaysLate": "past_due_90_count",
    "NumberRealEstateLoansOrLines": "real_estate_loan_count",
    "NumberOfTime60-89DaysPastDueNotWorse": "past_due_60_89_count",
    "NumberOfDependents": "dependent_count",
}

df = raw_df.rename(columns=rename_cols)

# timestamp 是公开镜像附加字段，不代表真实申请时间；后续不使用。
df = df.drop(columns=["timestamp"])

df.head()


## 3. 缺失值检查

缺失值本身可能有风险含义，所以先比较缺失客群和非缺失客群的坏客户率。


In [ ]:
income_missing = df["monthly_income"].isna()
dependent_missing = df["dependent_count"].isna()

missing_check = pd.DataFrame({
    "字段": ["monthly_income", "dependent_count"],
    "缺失数": [income_missing.sum(), dependent_missing.sum()],
    "缺失率": [income_missing.mean(), dependent_missing.mean()],
})

missing_check


In [ ]:
missing_bad_rate = pd.DataFrame({
    "客群": [
        "收入缺失",
        "收入不缺失",
        "家属数量缺失",
        "家属数量不缺失",
    ],
    "人数": [
        income_missing.sum(),
        (~income_missing).sum(),
        dependent_missing.sum(),
        (~dependent_missing).sum(),
    ],
    "坏客户率": [
        df.loc[income_missing, "is_bad_customer"].mean(),
        df.loc[~income_missing, "is_bad_customer"].mean(),
        df.loc[dependent_missing, "is_bad_customer"].mean(),
        df.loc[~dependent_missing, "is_bad_customer"].mean(),
    ],
})

missing_bad_rate


## 4. 逾期特殊编码检查

逾期次数中的 `96` 和 `98` 不应直接当作真实逾期次数。这里单独识别这些记录，并比较坏客户率。


In [ ]:
past_due_30_special = df["past_due_30_59_count"].isin([96, 98])
past_due_60_special = df["past_due_60_89_count"].isin([96, 98])
past_due_90_special = df["past_due_90_count"].isin([96, 98])

past_due_special = past_due_30_special | past_due_60_special | past_due_90_special

past_due_special_check = pd.DataFrame({
    "客群": ["逾期字段含 96/98", "逾期字段不含 96/98"],
    "人数": [past_due_special.sum(), (~past_due_special).sum()],
    "坏客户率": [
        df.loc[past_due_special, "is_bad_customer"].mean(),
        df.loc[~past_due_special, "is_bad_customer"].mean(),
    ],
})

past_due_special_check


In [ ]:
df.loc[
    past_due_special,
    [
        "customer_id",
        "is_bad_customer",
        "past_due_30_59_count",
        "past_due_60_89_count",
        "past_due_90_count",
    ],
].head(10)


## 5. 极端值检查

极端值不等于欺诈，但在风控里必须关注。它可能代表录入错误、特殊客户，或者真实的高风险行为。


In [ ]:
age_zero = df["age"].eq(0)
age_over_100 = df["age"].gt(100)
util_over_1 = df["credit_utilization"].gt(1)
util_over_10 = df["credit_utilization"].gt(10)
open_credit_over_40 = df["open_credit_count"].gt(40)
dependent_over_10 = df["dependent_count"].gt(10)

extreme_check = pd.DataFrame({
    "检查项": [
        "年龄为 0",
        "年龄大于 100",
        "额度使用率大于 1",
        "额度使用率大于 10",
        "开户/贷款账户数大于 40",
        "家属数量大于 10",
    ],
    "人数": [
        age_zero.sum(),
        age_over_100.sum(),
        util_over_1.sum(),
        util_over_10.sum(),
        open_credit_over_40.sum(),
        dependent_over_10.sum(),
    ],
    "坏客户率": [
        df.loc[age_zero, "is_bad_customer"].mean(),
        df.loc[age_over_100, "is_bad_customer"].mean(),
        df.loc[util_over_1, "is_bad_customer"].mean(),
        df.loc[util_over_10, "is_bad_customer"].mean(),
        df.loc[open_credit_over_40, "is_bad_customer"].mean(),
        df.loc[dependent_over_10, "is_bad_customer"].mean(),
    ],
})

extreme_check["占比"] = extreme_check["人数"] / len(df)
extreme_check


In [ ]:
df.loc[
    age_over_100 | open_credit_over_40 | dependent_over_10,
    [
        "customer_id",
        "is_bad_customer",
        "age",
        "monthly_income",
        "credit_utilization",
        "open_credit_count",
        "dependent_count",
    ],
].head(20)


## 6. 清洗规则

当前采用保守、实用的清洗方式：

- 年龄为 0：明显不合理，改成缺失；
- 逾期次数为 96/98：作为特殊编码，改成缺失；
- 收入缺失、家属数量缺失：先不填充，只增加缺失标记；
- 年龄大于 100、开户数大于 40、家属数量大于 10：保留原值，同时增加标记。


In [ ]:
clean_df = df.copy()

# 缺失值标记
clean_df["is_monthly_income_missing"] = clean_df["monthly_income"].isna().astype(int)
clean_df["is_dependent_count_missing"] = clean_df["dependent_count"].isna().astype(int)

# 年龄为 0：置为空，并保留标记
clean_df["is_age_zero"] = clean_df["age"].eq(0).astype(int)
clean_df.loc[clean_df["age"].eq(0), "age"] = np.nan

# 极端值标记
clean_df["is_age_over_100"] = clean_df["age"].gt(100).astype(int)
clean_df["is_credit_utilization_over_1"] = clean_df["credit_utilization"].gt(1).astype(int)
clean_df["is_open_credit_over_40"] = clean_df["open_credit_count"].gt(40).astype(int)
clean_df["is_dependent_over_10"] = clean_df["dependent_count"].gt(10).astype(int)

# 逾期特殊编码：置为空，并保留标记
clean_df["is_past_due_30_59_special"] = clean_df["past_due_30_59_count"].isin([96, 98]).astype(int)
clean_df["is_past_due_60_89_special"] = clean_df["past_due_60_89_count"].isin([96, 98]).astype(int)
clean_df["is_past_due_90_special"] = clean_df["past_due_90_count"].isin([96, 98]).astype(int)

clean_df.loc[clean_df["past_due_30_59_count"].isin([96, 98]), "past_due_30_59_count"] = np.nan
clean_df.loc[clean_df["past_due_60_89_count"].isin([96, 98]), "past_due_60_89_count"] = np.nan
clean_df.loc[clean_df["past_due_90_count"].isin([96, 98]), "past_due_90_count"] = np.nan

# 业务派生变量：是否曾经出现 30 天及以上逾期
clean_df["ever_dpd30_plus"] = (
    clean_df["past_due_30_59_count"].gt(0)
    | clean_df["past_due_60_89_count"].gt(0)
    | clean_df["past_due_90_count"].gt(0)
).astype(int)

clean_df.head()


## 7. 检查清洗是否生效


In [ ]:
past_due_special_after_cleaning = (
    clean_df["past_due_30_59_count"].isin([96, 98])
    | clean_df["past_due_60_89_count"].isin([96, 98])
    | clean_df["past_due_90_count"].isin([96, 98])
)

before_after_check = pd.DataFrame({
    "检查项": [
        "清洗前样本数",
        "清洗后样本数",
        "清洗前年龄为 0",
        "清洗后年龄为 0",
        "清洗前逾期字段含 96/98",
        "清洗后逾期字段含 96/98",
    ],
    "结果": [
        len(df),
        len(clean_df),
        df["age"].eq(0).sum(),
        clean_df["age"].eq(0).sum(),
        past_due_special.sum(),
        past_due_special_after_cleaning.sum(),
    ],
})

before_after_check


## 8. 保存清洗结果


In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
output_dir = PROJECT_ROOT / "output"

processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

clean_data_path = processed_dir / "credit_risk_cleaned.csv"
audit_summary_path = output_dir / "cleaning_audit_summary.csv"

missing_audit = missing_check.rename(columns={"字段": "检查项", "缺失数": "人数"})
past_due_audit = past_due_special_check.rename(columns={"客群": "检查项"})
extreme_audit = extreme_check[["检查项", "人数", "坏客户率", "占比"]]

cleaning_audit_summary = pd.concat(
    [missing_audit, past_due_audit, extreme_audit],
    ignore_index=True,
)

clean_df.to_csv(clean_data_path, index=False, encoding="utf-8-sig")
cleaning_audit_summary.to_csv(audit_summary_path, index=False, encoding="utf-8-sig")

clean_data_path, audit_summary_path


## 9. 本文件结论

- 清洗没有删除样本，只修正明显异常值并增加风险标记；
- 逾期 96/98 已改成缺失，避免把特殊编码当作真实逾期次数；
- 极端值没有直接删除，因为它们可能是真实风险信号；
- 后续分析应使用 `credit_risk_cleaned.csv`。

下一步：做风险因子分析，观察不同变量分组下的坏客户率。
